# Build SNP Nakeds. 
 - One Put Option per symbol

In [ ]:
## THIS CELL SHOULD BE IN ALL VSCODE NOTEBOOKS ##

MARKET = "SNP"

# Set the root
from from_root import from_root # type: ignore
ROOT = from_root()

import pandas as pd # type: ignore
from loguru import logger # type: ignore

pd.options.display.max_columns = None
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

from pathlib import Path
import sys

# Add `src` and ROOT to _src.pth in .venv to allow imports in VS Code
from sysconfig import get_path

if "src" not in Path.cwd().parts:
    src_path = str(Path(get_path("purelib")) / "_src.pth")
    with open(src_path, "w") as f:
        f.write(str(ROOT / "src\n"))
        f.write(str(ROOT))
        if str(ROOT) not in sys.path:
            sys.path.insert(1, str(ROOT))

# Start the Jupyter loop
from ib_async import util # type: ignore

util.startLoop()

logger.add(sink=ROOT / "log" / "ztest.log", mode="w")

In [ ]:
import itertools

import numpy as np
from ib_async import Contract, util

from ibfuncs import (MarketOrder, get_a_price_iv, get_ib, get_mkt_prices,
                     get_option_chains, make_chains, marginsAsync,
                     process_in_chunks, qualify_me)
from snp import assemble_snp_underlyings, snp_marcom, us_repo_rate
from utils import (append_black_scholes, append_safe_strikes, append_xPrice,
                   clean_ib_util_df, convert_to_utc_datetime,
                   get_closest_strike, get_dte, get_pickle, how_many_days_old,
                   load_config, pickle_me)

In [ ]:
# Set constants

config = load_config(MARKET=MARKET)

MAXDTE = config.get("MAXDTE")
PUTSTDMULT = config.get("PUTSTDMULT")
CALLSTDMULT = config.get("CALLSTDMULT")
MINEXPROM = config.get("MINEXPROM")

In [ ]:
# delete logs
from utils import overwrite_logs
# files = ROOT/'log'/'ztest.log'
overwrite_logs()

# Make unds

In [ ]:
df_unds = assemble_snp_underlyings(FRESH=True)

In [ ]:
# assemble the underlyings
unds_path = ROOT / 'data' / 'snp_unds.pkl'
if how_many_days_old(unds_path) > 0.35:
    df_unds = assemble_snp_underlyings(FRESH=True)
else:
    df_unds = get_pickle(unds_path)
df_unds.head()

# Make chains from the unds

In [ ]:
# make / get the chains

opts_path = ROOT/'data/'/'snp_opts.pkl'
days_old = how_many_days_old(opts_path)
if days_old is None or days_old >= 0.35:
    df_ch = make_chains(df_unds, save=True)
else:
    df_ch = get_pickle(opts_path)

df_ch.groupby('ib_symbol').first().head()

# Make PUT targets from chains - closest to strike

In [ ]:
dfp = df_ch[df_ch.right == "P"]
dfp = dfp[dfp.dte <= MAXDTE]

dfe = dfp[dfp.groupby(['ib_symbol', 'dte']).dte.transform('min').astype('int') == dfp.dte.astype('int')]

In [ ]:
# remove elements without undPrice
dfef = dfe[~dfe.undPrice.isnull()]
dfef

In [ ]:
dft = dfef.groupby(['ib_symbol', 'dte']) \
        .apply(lambda x: get_closest_strike(x), include_groups=False) \
        .reset_index().set_index('level_2') \
        .rename_axis('')

dft = dft.sort_values(['ib_symbol', 'dte'])

# Compute the IV to be average of historical and implied, if implied is less than historical

min_series = pd.Series(np.minimum(dft.und_iv, dft.und_hv))
weighted_avg_series = (dft.und_iv + dft.und_hv) / 2 * 0.75
iv = pd.Series(np.where(dft.und_iv < dft.und_hv, min_series + weighted_avg_series, dft.und_iv), index=dft.index)
dft = dft.assign(iv=iv)

# remove null ivs
dft = dft.loc[~dft.iv.isnull()]

# Get the safe strikes
dft = append_safe_strikes(dft, PUTSTDMULT, CALLSTDMULT)

# Make xPrice from black-scholes and market price

In [ ]:
# Get black scholes price

risk_free_rate = us_repo_rate() / 100
dft = append_black_scholes(dft, risk_free_rate)

In [ ]:
# Get the market prices
# ...build contracts

contracts = [Contract('OPT', symbol=s, lastTradeDateOrContractMonth=util.formatIBDatetime(e)[:8], strike=k, right=r, 
                    exchange='SMART', currency='USD') 
                    for s, e, k, r 
                    in zip(dft.ib_symbol, dft.expiry, dft.strike, dft.right)]


# ...qualify contracts
with get_ib(MARKET) as ib:
    cts = ib.run(process_in_chunks(ib, contracts, func=qualify_me, func_args={'desc': 'qualified'}, chunk_size=200, chunk_desc="Qualifying..."))


# Test get_a_price_iv

In [ ]:
# ...get market price for contracts
with get_ib(MARKET) as ib:
    res = ib.run(process_in_chunks(ib, cts, func=get_a_price_iv, func_args={'sleep': 15, 'gentick':''}, chunk_desc='Pricing'))

In [ ]:
df_price = pd.concat(res, ignore_index=True).drop(['secType', 'iv', 'hv'], axis=1)
dfn = df_price.merge(dft, on=['ib_symbol', 'expiry', 'strike', 'right'])
df = snp_marcom(dfn)
df = df[df.price>0] # remove zero price

In [ ]:
df_snp = append_xPrice(df.assign(lot=100), MINEXPROM)
df_snp = df_snp.reset_index(drop=True)
df_snp = df_snp.assign(lot=1) # reset lots for snp orders

In [ ]:
pickle_me(df_snp, ROOT/'data'/'snp_nakeds.pkl')